In [91]:
import os
import re
import pickle
from typing import Iterable

from utils import ViTSplit

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

import datasets
from datasets import load_from_disk, concatenate_datasets

from transformers import PreTrainedTokenizerFast
from tokenizers import Tokenizer, models, trainers, normalizers, pre_tokenizers, processors, Regex

In [92]:
torch.cuda.is_available()

False

In [93]:
DATASET_PATH = os.getenv("datasetPath")
ARTIFACTS_PATH = os.getenv("artifactsPath")
IMG_FOLDER_PATH = os.getenv("imgFolderPath")

In [94]:
main_dataset = load_from_disk(DATASET_PATH)

In [95]:
train_set = concatenate_datasets([main_dataset["train"], main_dataset["test"]])
val_set = main_dataset["valid"]
print(f"Training length: {len(train_set)}")
print(f"Validation length: {len(val_set)}")

Training length: 9397
Validation length: 976


In [96]:
type(train_set["image"][10])

PIL.JpegImagePlugin.JpegImageFile

In [97]:
class CharTokenizer:
    def __init__(self, iterable: Iterable[str]):
        self.iterable = iterable
        self.max_len = None
        self.fast_tokenizer = None
        self.tokenizer = None
    
    def get_max_len(self) -> int:
        """Returns the max length of encoded sequences"""
        assert self.fast_tokenizer != None, "Tokenizer is not trained"
        if self.max_len is None:
            the_max = 0
            for string in self.iterable["text"]:
                encoded = self.fast_tokenizer(string)["input_ids"]
                the_max = max(the_max, len(encoded))
            self.max_len = the_max
            return self.max_len
        else:
            return self.max_len

    def train(self) -> None:
        """Train tokenizer character wise"""
        tokenizer = Tokenizer(model = models.WordLevel())
        tokenizer.normalizer = normalizers.Sequence([normalizers.Lowercase(),
                                                     normalizers.Replace(Regex(r"[^a-z0-9\s]"), r""),
                                                     normalizers.Replace(Regex(r"\s+"), r" "),
                                                     normalizers.Strip()])
        
        tokenizer.pre_tokenizer = pre_tokenizers.Split(r"", "isolated", invert=False)
        tokenizer.post_processor = processors.TemplateProcessing(single=r"<sos> $0 <eos>",
                                                                 special_tokens=[(r"<sos>", 2), (r"<eos>", 3)])

        trainer = trainers.WordLevelTrainer(special_tokens=["<pad>", "<unk>", "<sos>", "<eos>"])
        tokenizer.train_from_iterator(iterator=self.iterable["text"], trainer=trainer)

        self.tokenizer = tokenizer
        self.fast_tokenizer = PreTrainedTokenizerFast(tokenizer_object = tokenizer)
        self.fast_tokenizer.pad_token = "<pad>"

In [98]:
with open(ARTIFACTS_PATH+r"\\char_tokenizer_obj.pkl", r"rb") as tokenizer_pickle_file:
    tokenizer = pickle.load(tokenizer_pickle_file)
tokenizer

In [99]:
with open(ARTIFACTS_PATH+r"\\statistics_dict.pkl", r"rb") as statistics_pickle_file:
    statistics = pickle.load(statistics_pickle_file)
statistics

{'mean': 0.9136844277381897, 'std': 0.14744582772254944}

In [100]:
input_ids = tokenizer.fast_tokenizer('hello world', padding='max_length', max_length=tokenizer.get_max_len())["input_ids"]
len(input_ids)

86

In [101]:
input_ids[:20]

[2, 13, 5, 14, 14, 8, 4, 20, 8, 12, 14, 15, 3, 0, 0, 0, 0, 0, 0, 0]

In [102]:
tokenizer.fast_tokenizer.decode(input_ids, skip_special_tokens=True)

'h e l l o   w o r l d'

In [103]:
ViTSplit(pic_size=(1, 128, 256), split_size=4)

In [123]:
class TrainDataSet(Dataset):
    def __init__(self, dataset: datasets.arrow_dataset.Dataset,
                 tokenizer: CharTokenizer,
                 transformations: transforms.transforms.Compose,
                 split: ViTSplit):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.transforms = transformations
        self.split = split
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, indx):
        row = self.dataset[indx]
        img = self.transforms(row["image"])
        splitted_img = self.split(img)
        target_inds = self.tokenizer.fast_tokenizer(row["text"],
                                                    padding='max_length',
                                                    max_length=self.tokenizer.get_max_len())["input_ids"]
        return torch.cat(splitted_img, dim=0), torch.tensor(target_inds, dtype=torch.long)

In [138]:
pytorch_train_set = TrainDataSet(dataset=train_set,
                         tokenizer=tokenizer,
                         transformations=transforms.Compose([transforms.ToTensor(),
                                                             transforms.Resize((128, 256)),
                                                             transforms.Normalize(mean=(statistics["mean"], ), std=(statistics["std"], ))]),
                        split=ViTSplit(pic_size=(1, 128, 256), split_size=4))